# 随机80/20留出与训练集5折CV模型比较

保留全部292炉，只比较预测模型；测试集不参与模型选择。

## 配置、依赖与自包含运行时

In [1]:
from pathlib import Path
import importlib.util
required = ['numpy','pandas','scipy','sklearn','joblib','openpyxl','lightgbm','catboost']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'缺少依赖: {missing}')
EXCEL_NAME = '4_month_data_2026_02_01_2026_06_25.xlsx'
EXCEL_PATH = next((root / EXCEL_NAME for root in [Path.cwd(), *Path.cwd().parents] if (root / EXCEL_NAME).exists()), None)
if EXCEL_PATH is None:
    raise FileNotFoundError(EXCEL_NAME)
PROJECT_ROOT = EXCEL_PATH.parent
OUTPUT_ROOT = PROJECT_ROOT / 'advanced_furnace_ml/random_cv_output'
ARTIFACTS_DIR = OUTPUT_ROOT / 'artifacts'
REPORTS_DIR = OUTPUT_ROOT / 'reports'
for directory in (OUTPUT_ROOT, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'excel': str(EXCEL_PATH), 'output': str(OUTPUT_ROOT)})

{'excel': '/Users/tian/Desktop/prediction_project/4_month_data_2026_02_01_2026_06_25.xlsx', 'output': '/Users/tian/Desktop/prediction_project/advanced_furnace_ml/random_cv_output'}


In [2]:
# 执行时只使用下面内嵌的源代码快照，不读取项目Python模块。
RUNTIME_SOURCES = {'__init__.py': '"""Advanced, isolated furnace gas prediction and recommendation experiments."""\n'
                '\n'
                '__version__ = "0.1.0"\n',
 'data.py': '"""Chronological Excel loading and immutable data contracts."""\n'
            '\n'
            'from pathlib import Path\n'
            '\n'
            'import pandas as pd\n'
            '\n'
            '\n'
            'TARGET_COL = "熔炼炉B当前批次总气耗_PLC"\n'
            'FEATURE_COLS = [\n'
            '    "10#熔炼炉总投料重量(kg)",\n'
            '    "10#熔炼炉固体料重量比例",\n'
            '    "熔炼炉B当前批次熔炼时间_PLC",\n'
            '    "熔炼炉B当前批次等待时长_PLC",\n'
            '    "熔炼炉B当前批次炉门打开次数_PLC",\n'
            '    "熔炼炉B当前批次炉门打开时长_PLC",\n'
            ']\n'
            'WEIGHT_COL, SOLID_COL, MELTING_COL, WAIT_COL, DOOR_COUNT_COL, DOOR_DURATION_COL = FEATURE_COLS\n'
            '\n'
            '\n'
            'def load_batch_data(path: str | Path, sheet_name: str = "Sheet1") -> pd.DataFrame:\n'
            '    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, engine="openpyxl")\n'
            '    batch_cols = [\n'
            '        col for col in raw.columns\n'
            '        if isinstance(raw.iloc[0, col], str) and raw.iloc[0, col].startswith("ER")\n'
            '    ]\n'
            '    variable_rows = [\n'
            '        row for row in raw.index\n'
            '        if isinstance(raw.iloc[row, 0], str) and raw.iloc[row, 0] != "Grand Total"\n'
            '    ]\n'
            '    values = raw.loc[variable_rows, batch_cols].apply(pd.to_numeric, errors="coerce").T\n'
            '    values.columns = raw.loc[variable_rows, 0].astype(str).tolist()\n'
            '    values = values.reset_index(drop=True)\n'
            '    values.insert(0, "batch_id", [str(raw.iloc[0, col]).strip() for col in batch_cols])\n'
            '    required = FEATURE_COLS + [TARGET_COL]\n'
            '    missing = [column for column in required if column not in values]\n'
            '    if missing:\n'
            '        raise ValueError(f"Excel 缺少建模列: {missing}")\n'
            '    if values["batch_id"].duplicated().any():\n'
            '        raise ValueError("Excel 包含重复炉次号。")\n'
            '    result = values[["batch_id"] + required].copy()\n'
            '    if result[TARGET_COL].isna().any():\n'
            '        raise ValueError("目标总气耗包含缺失值。")\n'
            '    return result\n'
            '\n'
            '\n'
            'def mark_target_outliers(df: pd.DataFrame) -> pd.DataFrame:\n'
            '    result = df.copy()\n'
            '    q1, q3 = result[TARGET_COL].quantile([0.25, 0.75])\n'
            '    result["is_high_gas_outlier"] = result[TARGET_COL] > q3 + 1.5 * (q3 - q1)\n'
            '    return result\n'
            '\n'
            '\n'
            'def chronological_dev_lock_split(df: pd.DataFrame, lock_size: int = 44) -> tuple[pd.DataFrame, '
            'pd.DataFrame]:\n'
            '    if len(df) <= lock_size:\n'
            '        raise ValueError("数据量不足以建立锁定测试集。")\n'
            '    split = len(df) - lock_size\n'
            '    return df.iloc[:split].copy(), df.iloc[split:].copy()\n',
 'experiment.py': '"""Development-only model selection and one-time locked audit."""\n'
                  '\n'
                  'from dataclasses import dataclass, field\n'
                  'import warnings\n'
                  '\n'
                  'import numpy as np\n'
                  'import pandas as pd\n'
                  'from sklearn.base import clone\n'
                  'from sklearn.exceptions import ConvergenceWarning\n'
                  'from sklearn.model_selection import ShuffleSplit\n'
                  '\n'
                  'from .data import FEATURE_COLS, TARGET_COL\n'
                  'from .models import (\n'
                  '    ResidualBoostRegressor,\n'
                  '    RouteRegressor,\n'
                  '    WeightedOOFEnsemble,\n'
                  '    build_base_models,\n'
                  '    build_tree_model_variants,\n'
                  '    fit_nonnegative_ensemble_weights,\n'
                  ')\n'
                  'from .uncertainty import conformal_radius, prediction_interval\n'
                  'from .validation import bootstrap_metric_interval, regression_metrics\n'
                  '\n'
                  '\n'
                  '@dataclass\n'
                  'class CandidateResult:\n'
                  '    name: str\n'
                  '    route: str\n'
                  '    estimator_template: object\n'
                  '    oof_predictions: np.ndarray\n'
                  '    fold_models: list\n'
                  '    fold_metrics: pd.DataFrame\n'
                  '    selection_score: float\n'
                  '    conformal_radius_90: float\n'
                  '\n'
                  '\n'
                  '@dataclass\n'
                  'class ModelExperiment:\n'
                  '    dev_df: pd.DataFrame\n'
                  '    results: dict[str, CandidateResult]\n'
                  '    summary: pd.DataFrame\n'
                  '    selected_name: str\n'
                  '    ensemble_members: list[str]\n'
                  '    ensemble_weights: np.ndarray\n'
                  '    tree_tuning_summary: pd.DataFrame\n'
                  '    frozen: bool = False\n'
                  '    full_dev_models: dict[str, object] = field(default_factory=dict)\n'
                  '\n'
                  '    def freeze(self):\n'
                  '        self.frozen = True\n'
                  '\n'
                  '\n'
                  'def evaluate_candidate(name, estimator, route, dev_df, splits) -> CandidateResult:\n'
                  '    X = dev_df[FEATURE_COLS]\n'
                  '    y = dev_df[TARGET_COL].to_numpy(dtype=float)\n'
                  '    oof = np.full(len(dev_df), np.nan)\n'
                  '    models = []\n'
                  '    rows = []\n'
                  '    for fold, (train_idx, valid_idx) in enumerate(splits):\n'
                  '        model = clone(estimator)\n'
                  '        with warnings.catch_warnings():\n'
                  '            warnings.simplefilter("ignore", ConvergenceWarning)\n'
                  '            model.fit(X.iloc[train_idx], y[train_idx])\n'
                  '        prediction = model.predict(X.iloc[valid_idx])\n'
                  '        oof[valid_idx] = prediction\n'
                  '        models.append(model)\n'
                  '        rows.append({\n'
                  '            "candidate": name,\n'
                  '            "route": route,\n'
                  '            "fold": fold,\n'
                  '            "train_size": len(train_idx),\n'
                  '            "valid_size": len(valid_idx),\n'
                  '            **regression_metrics(y[valid_idx], prediction),\n'
                  '        })\n'
                  '    metrics = pd.DataFrame(rows)\n'
                  '    score = float(metrics["rmse"].mean() + 0.25 * metrics["rmse"].std())\n'
                  '    return CandidateResult(\n'
                  '        name=name,\n'
                  '        route=route,\n'
                  '        estimator_template=estimator,\n'
                  '        oof_predictions=oof,\n'
                  '        fold_models=models,\n'
                  '        fold_metrics=metrics,\n'
                  '        selection_score=score,\n'
                  '        conformal_radius_90=conformal_radius(y, oof, 0.90),\n'
                  '    )\n'
                  '\n'
                  '\n'
                  'def _summary_row(result: CandidateResult):\n'
                  '    metrics = result.fold_metrics\n'
                  '    return {\n'
                  '        "candidate": result.name,\n'
                  '        "route": result.route,\n'
                  '        "mae_mean": metrics["mae"].mean(),\n'
                  '        "rmse_mean": metrics["rmse"].mean(),\n'
                  '        "rmse_std": metrics["rmse"].std(),\n'
                  '        "rmse_worst": metrics["rmse"].max(),\n'
                  '        "wape_mean": metrics["wape"].mean(),\n'
                  '        "r2_mean": metrics["r2"].mean(),\n'
                  '        "error_gt_10pct_rate": metrics["error_gt_10pct_rate"].mean(),\n'
                  '        "selection_score": result.selection_score,\n'
                  '        "conformal_radius_90": result.conformal_radius_90,\n'
                  '    }\n'
                  '\n'
                  '\n'
                  'def run_model_matrix(dev_df, splits, model_names=None, routes=("direct", "unit"), tune_trees: bool '
                  '= True) -> ModelExperiment:\n'
                  '    base_models = build_base_models()\n'
                  '    names = tuple(base_models) if model_names is None else tuple(model_names)\n'
                  '    tuning_rows = []\n'
                  '    if tune_trees:\n'
                  '        variants = build_tree_model_variants()\n'
                  '        for tree_name in ("LightGBM", "CatBoost"):\n'
                  '            if tree_name not in names:\n'
                  '                continue\n'
                  '            scored_variants = []\n'
                  '            for label, template in variants[tree_name].items():\n'
                  '                variant_result = evaluate_candidate(\n'
                  '                    f"tuning__{tree_name}__{label}",\n'
                  '                    RouteRegressor(template, "direct"),\n'
                  '                    "direct",\n'
                  '                    dev_df,\n'
                  '                    splits,\n'
                  '                )\n'
                  '                scored_variants.append((variant_result.selection_score, label, template))\n'
                  '                tuning_rows.append({\n'
                  '                    "model": tree_name,\n'
                  '                    "variant": label,\n'
                  '                    "selection_score": variant_result.selection_score,\n'
                  '                    "rmse_mean": variant_result.fold_metrics["rmse"].mean(),\n'
                  '                    "rmse_std": variant_result.fold_metrics["rmse"].std(),\n'
                  '                })\n'
                  '            _, selected_label, selected_template = min(scored_variants, key=lambda item: item[0])\n'
                  '            base_models[tree_name] = selected_template\n'
                  '            for row in tuning_rows:\n'
                  '                if row["model"] == tree_name:\n'
                  '                    row["selected"] = row["variant"] == selected_label\n'
                  '        if "LightGBM" in names:\n'
                  '            base_models["Ridge+LGBMResidual"] = ResidualBoostRegressor(\n'
                  '                base_models["Ridge"], base_models["LightGBM"]\n'
                  '            )\n'
                  '            base_models["Huber+LGBMResidual"] = ResidualBoostRegressor(\n'
                  '                base_models["Huber"], base_models["LightGBM"]\n'
                  '            )\n'
                  '    results = {}\n'
                  '    for model_name in names:\n'
                  '        for route in routes:\n'
                  '            candidate_name = f"{model_name}__{route}"\n'
                  '            estimator = RouteRegressor(base_models[model_name], route)\n'
                  '            results[candidate_name] = evaluate_candidate(candidate_name, estimator, route, dev_df, '
                  'splits)\n'
                  '\n'
                  '    ranked = sorted(results.values(), key=lambda item: item.selection_score)\n'
                  '    ensemble_sources = ranked[: min(4, len(ranked))]\n'
                  '    valid = np.logical_and.reduce([np.isfinite(item.oof_predictions) for item in '
                  'ensemble_sources])\n'
                  '    matrix = np.column_stack([item.oof_predictions[valid] for item in ensemble_sources])\n'
                  '    y_valid = dev_df[TARGET_COL].to_numpy(dtype=float)[valid]\n'
                  '    weights = fit_nonnegative_ensemble_weights(matrix, y_valid)\n'
                  '    ensemble = WeightedOOFEnsemble(\n'
                  '        [item.estimator_template for item in ensemble_sources], weights\n'
                  '    )\n'
                  '    results["OOFEnsemble"] = evaluate_candidate(\n'
                  '        "OOFEnsemble", ensemble, "mixed_total", dev_df, splits\n'
                  '    )\n'
                  '    summary = pd.DataFrame([_summary_row(result) for result in results.values()])\n'
                  '    summary = summary.sort_values("selection_score").reset_index(drop=True)\n'
                  '    return ModelExperiment(\n'
                  '        dev_df=dev_df.copy(),\n'
                  '        results=results,\n'
                  '        summary=summary,\n'
                  '        selected_name=str(summary.iloc[0]["candidate"]),\n'
                  '        ensemble_members=[item.name for item in ensemble_sources],\n'
                  '        ensemble_weights=weights,\n'
                  '        tree_tuning_summary=pd.DataFrame(tuning_rows),\n'
                  '    )\n'
                  '\n'
                  '\n'
                  'def random_cv_reference(experiment: ModelExperiment, n_splits: int = 3, seed: int = 42):\n'
                  '    X = experiment.dev_df[FEATURE_COLS]\n'
                  '    y = experiment.dev_df[TARGET_COL].to_numpy(dtype=float)\n'
                  '    splitter = ShuffleSplit(n_splits=n_splits, test_size=.20, random_state=seed)\n'
                  '    rows = []\n'
                  '    for name, result in experiment.results.items():\n'
                  '        for split_id, (train_idx, valid_idx) in enumerate(splitter.split(X)):\n'
                  '            model = clone(result.estimator_template)\n'
                  '            with warnings.catch_warnings():\n'
                  '                warnings.simplefilter("ignore", ConvergenceWarning)\n'
                  '                model.fit(X.iloc[train_idx], y[train_idx])\n'
                  '            prediction = model.predict(X.iloc[valid_idx])\n'
                  '            rows.append({\n'
                  '                "candidate": name,\n'
                  '                "split": split_id,\n'
                  '                "train_size": len(train_idx),\n'
                  '                "valid_size": len(valid_idx),\n'
                  '                **regression_metrics(y[valid_idx], prediction),\n'
                  '            })\n'
                  '    return pd.DataFrame(rows)\n'
                  '\n'
                  '\n'
                  'def audit_frozen_models(experiment: ModelExperiment, locked_df: pd.DataFrame) -> pd.DataFrame:\n'
                  '    if not experiment.frozen:\n'
                  '        raise RuntimeError("Model experiment must freeze() before locked audit.")\n'
                  '    X_dev = experiment.dev_df[FEATURE_COLS]\n'
                  '    y_dev = experiment.dev_df[TARGET_COL]\n'
                  '    X_lock = locked_df[FEATURE_COLS]\n'
                  '    y_lock = locked_df[TARGET_COL].to_numpy(dtype=float)\n'
                  '    rows = []\n'
                  '    for name, result in experiment.results.items():\n'
                  '        model = clone(result.estimator_template)\n'
                  '        with warnings.catch_warnings():\n'
                  '            warnings.simplefilter("ignore", ConvergenceWarning)\n'
                  '            model.fit(X_dev, y_dev)\n'
                  '        prediction = model.predict(X_lock)\n'
                  '        try:\n'
                  '            native_std_mean = float(np.mean(model.predict_std(X_lock)))\n'
                  '        except (AttributeError, TypeError):\n'
                  '            native_std_mean = np.nan\n'
                  '        experiment.full_dev_models[name] = model\n'
                  '        low, high = prediction_interval(prediction, result.conformal_radius_90)\n'
                  '        metrics = regression_metrics(y_lock, prediction)\n'
                  '        rmse_low, rmse_high = bootstrap_metric_interval(y_lock, prediction, "rmse", seed=42, '
                  'n_boot=500)\n'
                  '        rows.append({\n'
                  '            "candidate": name,\n'
                  '            "route": result.route,\n'
                  '            **metrics,\n'
                  '            "rmse_bootstrap_low95": rmse_low,\n'
                  '            "rmse_bootstrap_high95": rmse_high,\n'
                  '            "interval_coverage_90": float(((y_lock >= low) & (y_lock <= high)).mean()),\n'
                  '            "interval_mean_width": float(np.mean(high - low)),\n'
                  '            "native_model_std_mean": native_std_mean,\n'
                  '            "selected_before_lock": name == experiment.selected_name,\n'
                  '        })\n'
                  '    return pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)\n',
 'features.py': '"""Leakage-safe furnace feature engineering and target routes."""\n'
                '\n'
                'import numpy as np\n'
                'import pandas as pd\n'
                'from sklearn.base import BaseEstimator, TransformerMixin\n'
                '\n'
                'from .data import (\n'
                '    DOOR_COUNT_COL,\n'
                '    DOOR_DURATION_COL,\n'
                '    FEATURE_COLS,\n'
                '    MELTING_COL,\n'
                '    SOLID_COL,\n'
                '    TARGET_COL,\n'
                '    WAIT_COL,\n'
                '    WEIGHT_COL,\n'
                ')\n'
                '\n'
                '\n'
                'DERIVED_COLS = [\n'
                '    "derived__solid_weight_kg",\n'
                '    "derived__avg_door_duration",\n'
                '    "derived__door_count_per_melting_time",\n'
                '    "derived__door_duration_per_melting_time",\n'
                '    "derived__waiting_ratio",\n'
                '    "derived__non_waiting_melting_time",\n'
                ']\n'
                '\n'
                '\n'
                'class FurnaceFeatureEngineer(BaseEstimator, TransformerMixin):\n'
                '    def fit(self, X, y=None):\n'
                '        self.feature_names_in_ = np.asarray(FEATURE_COLS, dtype=object)\n'
                '        return self\n'
                '\n'
                '    def transform(self, X):\n'
                '        frame = pd.DataFrame(X).copy()[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")\n'
                '        safe_melting = frame[MELTING_COL].replace(0, np.nan)\n'
                '        safe_count = frame[DOOR_COUNT_COL].replace(0, np.nan)\n'
                '        derived = pd.DataFrame(index=frame.index)\n'
                '        derived[DERIVED_COLS[0]] = frame[WEIGHT_COL] * frame[SOLID_COL] / 100.0\n'
                '        derived[DERIVED_COLS[1]] = frame[DOOR_DURATION_COL] / safe_count\n'
                '        derived[DERIVED_COLS[2]] = frame[DOOR_COUNT_COL] / safe_melting\n'
                '        derived[DERIVED_COLS[3]] = frame[DOOR_DURATION_COL] / safe_melting\n'
                '        derived[DERIVED_COLS[4]] = frame[WAIT_COL] / safe_melting\n'
                '        derived[DERIVED_COLS[5]] = frame[MELTING_COL] - frame[WAIT_COL]\n'
                '        derived = derived.replace([np.inf, -np.inf], np.nan)\n'
                '        return pd.concat([frame, derived], axis=1)\n'
                '\n'
                '    def get_feature_names_out(self, input_features=None):\n'
                '        return np.asarray(FEATURE_COLS + DERIVED_COLS, dtype=object)\n'
                '\n'
                '\n'
                'def _weight_tonnes(frame: pd.DataFrame, median: float | None = None) -> pd.Series:\n'
                '    weight = pd.to_numeric(frame[WEIGHT_COL], errors="coerce")\n'
                '    fill = float(weight.median()) if median is None else float(median)\n'
                '    return weight.fillna(fill) / 1000.0\n'
                '\n'
                '\n'
                'def target_for_route(df: pd.DataFrame, route: str) -> pd.Series:\n'
                '    target = pd.to_numeric(df[TARGET_COL], errors="coerce")\n'
                '    if route == "direct":\n'
                '        return target\n'
                '    if route == "unit":\n'
                '        return target / _weight_tonnes(df)\n'
                '    raise ValueError(f"未知目标路线: {route}")\n'
                '\n'
                '\n'
                'def prediction_to_total_gas(prediction, X: pd.DataFrame, route: str, weight_median: float | None = '
                'None) -> np.ndarray:\n'
                '    values = np.asarray(prediction, dtype=float)\n'
                '    if route == "direct":\n'
                '        return values\n'
                '    if route == "unit":\n'
                '        return values * _weight_tonnes(X, weight_median).to_numpy()\n'
                '    raise ValueError(f"未知目标路线: {route}")\n',
 'models.py': '"""Small-data candidate regressors, target routes, residuals, and ensembles."""\n'
              '\n'
              'from copy import deepcopy\n'
              '\n'
              'import numpy as np\n'
              'import pandas as pd\n'
              'from catboost import CatBoostRegressor\n'
              'from lightgbm import LGBMRegressor\n'
              'from scipy.optimize import minimize\n'
              'from sklearn.base import BaseEstimator, RegressorMixin, clone\n'
              'from sklearn.gaussian_process import GaussianProcessRegressor\n'
              'from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel\n'
              'from sklearn.impute import SimpleImputer\n'
              'from sklearn.linear_model import ElasticNet, HuberRegressor, Ridge\n'
              'from sklearn.pipeline import Pipeline\n'
              'from sklearn.preprocessing import RobustScaler, SplineTransformer, StandardScaler\n'
              '\n'
              'from .data import FEATURE_COLS, WEIGHT_COL\n'
              'from .features import FurnaceFeatureEngineer, prediction_to_total_gas\n'
              '\n'
              '\n'
              'def _imputer():\n'
              '    return SimpleImputer(strategy="median").set_output(transform="pandas")\n'
              '\n'
              '\n'
              'def _numeric_pipeline(model, scale="standard"):\n'
              '    steps = [("features", FurnaceFeatureEngineer()), ("imputer", _imputer())]\n'
              '    if scale == "standard":\n'
              '        steps.append(("scaler", StandardScaler()))\n'
              '    elif scale == "robust":\n'
              '        steps.append(("scaler", RobustScaler()))\n'
              '    steps.append(("model", model))\n'
              '    return Pipeline(steps)\n'
              '\n'
              '\n'
              'class ResidualBoostRegressor(BaseEstimator, RegressorMixin):\n'
              '    def __init__(self, base_model, residual_model):\n'
              '        self.base_model = base_model\n'
              '        self.residual_model = residual_model\n'
              '\n'
              '    def fit(self, X, y):\n'
              '        self.base_model_ = clone(self.base_model).fit(X, y)\n'
              '        residual = np.asarray(y, dtype=float) - self.base_model_.predict(X)\n'
              '        self.residual_model_ = clone(self.residual_model).fit(X, residual)\n'
              '        return self\n'
              '\n'
              '    def predict(self, X):\n'
              '        return self.base_model_.predict(X) + self.residual_model_.predict(X)\n'
              '\n'
              '\n'
              'class RouteRegressor(BaseEstimator, RegressorMixin):\n'
              '    def __init__(self, model, route="direct"):\n'
              '        self.model = model\n'
              '        self.route = route\n'
              '\n'
              '    def fit(self, X, y):\n'
              '        frame = pd.DataFrame(X).copy()\n'
              '        self.weight_median_ = float(pd.to_numeric(frame[WEIGHT_COL], errors="coerce").median())\n'
              '        target = np.asarray(y, dtype=float)\n'
              '        if self.route == "unit":\n'
              '            tonnes = pd.to_numeric(frame[WEIGHT_COL], '
              'errors="coerce").fillna(self.weight_median_).to_numpy() / 1000.0\n'
              '            target = target / tonnes\n'
              '        elif self.route != "direct":\n'
              '            raise ValueError(f"未知目标路线: {self.route}")\n'
              '        self.model_ = clone(self.model).fit(frame[FEATURE_COLS], target)\n'
              '        return self\n'
              '\n'
              '    def predict(self, X):\n'
              '        frame = pd.DataFrame(X).copy()[FEATURE_COLS]\n'
              '        prediction = self.model_.predict(frame)\n'
              '        return prediction_to_total_gas(prediction, frame, self.route, self.weight_median_)\n'
              '\n'
              '    def predict_std(self, X):\n'
              '        frame = pd.DataFrame(X).copy()[FEATURE_COLS]\n'
              '        if not isinstance(self.model_, Pipeline):\n'
              '            raise TypeError("底层模型不支持原生预测标准差。")\n'
              '        transformed = self.model_[:-1].transform(frame)\n'
              '        final_model = self.model_.steps[-1][1]\n'
              '        if not isinstance(final_model, GaussianProcessRegressor):\n'
              '            raise TypeError("底层模型不支持原生预测标准差。")\n'
              '        _, standard_deviation = final_model.predict(transformed, return_std=True)\n'
              '        return prediction_to_total_gas(\n'
              '            standard_deviation, frame, self.route, self.weight_median_\n'
              '        )\n'
              '\n'
              '\n'
              'class WeightedOOFEnsemble(BaseEstimator, RegressorMixin):\n'
              '    def __init__(self, models, weights):\n'
              '        self.models = models\n'
              '        self.weights = weights\n'
              '\n'
              '    def fit(self, X, y):\n'
              '        self.models_ = [clone(model).fit(X, y) for model in self.models]\n'
              '        return self\n'
              '\n'
              '    def predict(self, X):\n'
              '        members = getattr(self, "models_", self.models)\n'
              '        matrix = np.column_stack([model.predict(X) for model in members])\n'
              '        return matrix @ np.asarray(self.weights, dtype=float)\n'
              '\n'
              '\n'
              'def fit_nonnegative_ensemble_weights(oof_matrix, y):\n'
              '    matrix = np.asarray(oof_matrix, dtype=float)\n'
              '    target = np.asarray(y, dtype=float)\n'
              '    count = matrix.shape[1]\n'
              '    result = minimize(\n'
              '        lambda weights: np.mean((matrix @ weights - target) ** 2),\n'
              '        np.repeat(1.0 / count, count),\n'
              '        bounds=[(0.0, 1.0)] * count,\n'
              '        constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1.0},\n'
              '        method="SLSQP",\n'
              '    )\n'
              '    if not result.success:\n'
              '        return np.repeat(1.0 / count, count)\n'
              '    weights = np.maximum(result.x, 0)\n'
              '    return weights / weights.sum()\n'
              '\n'
              '\n'
              'def build_base_models(seed: int = 42):\n'
              '    ridge = _numeric_pipeline(Ridge(alpha=10.0))\n'
              '    huber = _numeric_pipeline(HuberRegressor(epsilon=1.5, max_iter=3000), scale="robust")\n'
              '    lgbm = _numeric_pipeline(\n'
              '        LGBMRegressor(\n'
              '            n_estimators=350, learning_rate=0.03, num_leaves=7, max_depth=3,\n'
              '            min_child_samples=25, max_bin=31, reg_alpha=1.0, reg_lambda=10.0,\n'
              '            subsample=0.85, colsample_bytree=0.85, random_state=seed,\n'
              '            n_jobs=-1, verbose=-1,\n'
              '        ),\n'
              '        scale=None,\n'
              '    )\n'
              '    models = {\n'
              '        "Ridge": ridge,\n'
              '        "ElasticNet": _numeric_pipeline(ElasticNet(alpha=0.05, l1_ratio=0.25, max_iter=10000)),\n'
              '        "Huber": huber,\n'
              '        "GAM": Pipeline([\n'
              '            ("features", FurnaceFeatureEngineer()),\n'
              '            ("imputer", _imputer()),\n'
              '            ("splines", SplineTransformer(n_knots=4, degree=2, include_bias=False)),\n'
              '            ("scaler", StandardScaler()),\n'
              '            ("model", Ridge(alpha=10.0)),\n'
              '        ]),\n'
              '        "GPR": _numeric_pipeline(\n'
              '            GaussianProcessRegressor(\n'
              '                kernel=ConstantKernel(1.0) * Matern(length_scale=1.0, nu=1.5) + '
              'WhiteKernel(noise_level=0.2),\n'
              '                alpha=1e-6, normalize_y=True, n_restarts_optimizer=0, random_state=seed,\n'
              '            )\n'
              '        ),\n'
              '        "CatBoost": _numeric_pipeline(\n'
              '            CatBoostRegressor(\n'
              '                iterations=300, depth=4, learning_rate=0.03, loss_function="RMSE",\n'
              '                l2_leaf_reg=8.0, random_seed=seed, verbose=False, allow_writing_files=False,\n'
              '            ),\n'
              '            scale=None,\n'
              '        ),\n'
              '        "LightGBM": lgbm,\n'
              '    }\n'
              '    models["Ridge+LGBMResidual"] = ResidualBoostRegressor(ridge, lgbm)\n'
              '    models["Huber+LGBMResidual"] = ResidualBoostRegressor(huber, lgbm)\n'
              '    return models\n'
              '\n'
              '\n'
              'def build_tree_model_variants(seed: int = 42):\n'
              '    base = build_base_models(seed)\n'
              '    lgbm_variants = {}\n'
              '    for label, params in {\n'
              '        "very_small": {"num_leaves": 4, "max_depth": 2, "min_child_samples": 30, "reg_lambda": 15.0},\n'
              '        "small": {"num_leaves": 7, "max_depth": 3, "min_child_samples": 25, "reg_lambda": 10.0},\n'
              '        "medium_small": {"num_leaves": 10, "max_depth": 4, "min_child_samples": 20, "reg_lambda": '
              '8.0},\n'
              '    }.items():\n'
              '        lgbm_variants[label] = clone(base["LightGBM"]).set_params(\n'
              '            **{f"model__{key}": value for key, value in params.items()}\n'
              '        )\n'
              '    cat_variants = {}\n'
              '    for label, params in {\n'
              '        "depth3": {"depth": 3, "l2_leaf_reg": 10.0},\n'
              '        "depth4": {"depth": 4, "l2_leaf_reg": 8.0},\n'
              '    }.items():\n'
              '        cat_variants[label] = clone(base["CatBoost"]).set_params(\n'
              '            **{f"model__{key}": value for key, value in params.items()}\n'
              '        )\n'
              '    return {"LightGBM": lgbm_variants, "CatBoost": cat_variants}\n',
 'uncertainty.py': '"""Conformal intervals and chronological fold disagreement."""\n'
                   '\n'
                   'import numpy as np\n'
                   '\n'
                   '\n'
                   'def conformal_radius(y_true, oof_prediction, coverage: float = 0.90) -> float:\n'
                   '    y = np.asarray(y_true, dtype=float)\n'
                   '    prediction = np.asarray(oof_prediction, dtype=float)\n'
                   '    valid = np.isfinite(prediction)\n'
                   '    if not valid.any():\n'
                   '        raise ValueError("没有可用于 conformal 校准的 OOF 预测。")\n'
                   '    return float(np.quantile(np.abs(y[valid] - prediction[valid]), coverage))\n'
                   '\n'
                   '\n'
                   'def prediction_interval(prediction, radius: float):\n'
                   '    values = np.asarray(prediction, dtype=float)\n'
                   '    return np.maximum(0.0, values - float(radius)), values + float(radius)\n'
                   '\n'
                   '\n'
                   'def fold_prediction_summary(fold_models, X):\n'
                   '    matrix = np.vstack([model.predict(X) for model in fold_models])\n'
                   '    return {\n'
                   '        "matrix": matrix,\n'
                   '        "median": np.median(matrix, axis=0),\n'
                   '        "mean": matrix.mean(axis=0),\n'
                   '        "std": matrix.std(axis=0),\n'
                   '    }\n',
 'validation.py': '"""Large-window chronological validation and locked audit helpers."""\n'
                  '\n'
                  'from dataclasses import dataclass\n'
                  '\n'
                  'import numpy as np\n'
                  'from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n'
                  '\n'
                  '\n'
                  'def large_window_splits(n_dev: int = 248):\n'
                  '    if n_dev != 248:\n'
                  '        raise ValueError("当前正式设计要求开发区恰好为248炉。")\n'
                  '    boundaries = [(148, 181), (181, 214), (214, 248)]\n'
                  '    return [(np.arange(train_end), np.arange(train_end, test_end)) for train_end, test_end in '
                  'boundaries]\n'
                  '\n'
                  '\n'
                  'def regression_metrics(y_true, y_pred) -> dict:\n'
                  '    y = np.asarray(y_true, dtype=float)\n'
                  '    p = np.asarray(y_pred, dtype=float)\n'
                  '    absolute = np.abs(y - p)\n'
                  '    denominator = np.maximum(np.abs(y), 1e-12)\n'
                  '    return {\n'
                  '        "mae": float(mean_absolute_error(y, p)),\n'
                  '        "rmse": float(mean_squared_error(y, p) ** 0.5),\n'
                  '        "wape": float(absolute.sum() / denominator.sum()),\n'
                  '        "r2": float(r2_score(y, p)),\n'
                  '        "error_gt_10pct_rate": float((absolute / denominator > 0.10).mean()),\n'
                  '    }\n'
                  '\n'
                  '\n'
                  'def bootstrap_metric_interval(y_true, y_pred, metric: str, seed: int, n_boot: int = 1000):\n'
                  '    y = np.asarray(y_true, dtype=float)\n'
                  '    p = np.asarray(y_pred, dtype=float)\n'
                  '    rng = np.random.default_rng(seed)\n'
                  '    values = []\n'
                  '    for _ in range(n_boot):\n'
                  '        index = rng.integers(0, len(y), len(y))\n'
                  '        values.append(regression_metrics(y[index], p[index])[metric])\n'
                  '    return float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))\n'
                  '\n'
                  '\n'
                  '@dataclass\n'
                  'class LockedAudit:\n'
                  '    _values: object\n'
                  '    _frozen: bool = False\n'
                  '\n'
                  '    def freeze(self):\n'
                  '        self._frozen = True\n'
                  '\n'
                  '    def values(self):\n'
                  '        if not self._frozen:\n'
                  '            raise RuntimeError("Locked audit cannot be read before freeze().")\n'
                  '        return self._values\n'}

import importlib
import sys
RUNTIME_ROOT = OUTPUT_ROOT / 'runtime'
PACKAGE_DIR = RUNTIME_ROOT / 'random_cv_runtime'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
for filename, source_text in RUNTIME_SOURCES.items():
    (PACKAGE_DIR / filename).write_text(source_text, encoding='utf-8')
if str(RUNTIME_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNTIME_ROOT))
importlib.invalidate_caches()
print(f'独立运行时已生成: {PACKAGE_DIR}')

独立运行时已生成: /Users/tian/Desktop/prediction_project/advanced_furnace_ml/random_cv_output/runtime/random_cv_runtime


## Excel数据检查：保留全部292炉

In [3]:
import numpy as np
import pandas as pd
from random_cv_runtime.data import FEATURE_COLS, TARGET_COL, load_batch_data, mark_target_outliers
df = mark_target_outliers(load_batch_data(EXCEL_PATH))
assert len(df) == 292
outlier_count_before_split = int(df['is_high_gas_outlier'].sum())
display(df.head())
display({'总炉次': len(df), '异常炉次标记数（保留）': outlier_count_before_split, '删除炉次': 0})

,batch_id,10#熔炼炉总投料重量(kg),10#熔炼炉固体料重量比例,熔炼炉B当前批次熔炼时间_PLC,熔炼炉B当前批次等待时长_PLC,熔炼炉B当前批次炉门打开次数_PLC,熔炼炉B当前批次炉门打开时长_PLC,熔炼炉B当前批次总气耗_PLC,is_high_gas_outlier
0,ER014,91920.0,54.47,11.75,1.67,13.0,71.0,5190.63,False
1,ER015,76932.0,31.77,8.37,0.97,10.0,149.0,2878.56,False
2,ER016,97523.0,47.40,10.59,1.48,12.0,75.0,4395.48,False
3,ER017,89641.0,41.13,8.93,2.95,12.0,79.0,3744.03,False
4,ER018,NaN,NaN,8.25,0.77,12.0,36.0,2993.34,False


{'总炉次': 292, '异常炉次标记数（保留）': 5, '删除炉次': 0}

## 随机训练集与测试集

In [4]:
from sklearn.model_selection import KFold, train_test_split
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
assert len(train_df) == 233
assert len(test_df) == 59
outlier_count_after_split = int(train_df['is_high_gas_outlier'].sum() + test_df['is_high_gas_outlier'].sum())
assert outlier_count_after_split == outlier_count_before_split
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
random_splits = list(kfold.split(train_df))
assert len(random_splits) == 5
for train_index, valid_index in random_splits:
    assert set(train_index).isdisjoint(set(valid_index))
    assert int(max(train_index.max(), valid_index.max())) < len(train_df)
split_table = pd.DataFrame([
    {'fold': i + 1, 'train_size': len(tr), 'valid_size': len(va), 'overlap': len(set(tr) & set(va))}
    for i, (tr, va) in enumerate(random_splits)
])
display({'训练炉次': len(train_df), '测试炉次': len(test_df), 'random_state': 42})
display(split_table)

{'训练炉次': 233, '测试炉次': 59, 'random_state': 42}

,fold,train_size,valid_size,overlap
0,1,186,47,0
1,2,186,47,0
2,3,186,47,0
3,4,187,46,0
4,5,187,46,0


## 训练集内部5折交叉验证与19候选

LightGBM/CatBoost小数据参数、OOF权重和Champion都只使用233炉训练集的五折结果。

In [5]:
from random_cv_runtime.experiment import run_model_matrix
random_experiment = run_model_matrix(train_df, random_splits)
random_cv_summary = random_experiment.summary.copy().sort_values(
    ['selection_score', 'mae_mean', 'candidate']
).reset_index(drop=True)
selected_before_test = str(random_cv_summary.iloc[0]['candidate'])
random_experiment.selected_name = selected_before_test
random_cv_summary['selected_before_test'] = random_cv_summary['candidate'].eq(selected_before_test)
random_cv_fold_metrics = pd.concat(
    [result.fold_metrics for result in random_experiment.results.values()], ignore_index=True
)
assert random_cv_summary['candidate'].nunique() == 19
assert len(random_experiment.ensemble_weights) == 4
assert np.all(random_experiment.ensemble_weights >= 0)
assert abs(float(random_experiment.ensemble_weights.sum()) - 1.0) < 1e-10
ensemble_table = pd.DataFrame({
    'member': random_experiment.ensemble_members, 'weight': random_experiment.ensemble_weights
})
random_experiment.freeze()
display(random_cv_summary.round(4))
display(random_experiment.tree_tuning_summary.round(4))
display(ensemble_table.round(4))
print(f'CV Champion（测试集尚未查看）: {selected_before_test}')

,candidate,route,mae_mean,rmse_mean,rmse_std,rmse_worst,wape_mean,r2_mean,error_gt_10pct_rate,selection_score,conformal_radius_90,selected_before_test
0,Ridge__direct,direct,302.6455,400.3482,57.4475,474.5989,0.0835,0.7992,0.3092,414.7101,630.1393,True
1,ElasticNet__direct,direct,303.5668,402.3157,55.7575,474.6506,0.0837,0.7971,0.3092,416.2550,631.3108,False
2,OOFEnsemble,mixed_total,303.1860,405.8547,55.9327,481.8874,0.0836,0.7953,0.3134,419.8379,635.3048,False
3,Ridge+LGBMResidual__direct,direct,302.6640,412.4934,62.8721,504.6022,0.0833,0.7917,0.3130,428.2114,649.4725,False
4,Ridge__unit,unit,340.8925,453.5601,45.2749,506.5703,0.0940,0.7379,0.3992,464.8789,670.5988,False
5,Ridge+LGBMResidual__unit,unit,322.4622,453.9626,58.2886,512.9324,0.0889,0.7333,0.3129,468.5347,643.8833,False
6,GPR__direct,direct,295.0731,434.1085,138.4458,674.7157,0.0812,0.7886,0.2959,468.7199,624.7679,False
7,ElasticNet__unit,unit,342.7942,458.5878,43.3705,513.2866,0.0945,0.7319,0.3992,469.4304,668.7698,False
8,Huber__direct,direct,306.5335,453.1263,78.3276,533.8706,0.0844,0.7427,0.2875,472.7082,619.2889,False
9,Huber+LGBMResidual__direct,direct,318.6294,460.7711,95.7990,568.0896,0.0877,0.7348,0.3349,484.7208,649.9430,False


,model,variant,selection_score,rmse_mean,rmse_std,selected
0,LightGBM,very_small,735.6065,657.9484,310.6325,False
1,LightGBM,small,691.3544,609.2444,328.4402,False
2,LightGBM,medium_small,670.9443,596.4473,297.9880,True
3,CatBoost,depth3,715.5695,654.1313,245.7527,False
4,CatBoost,depth4,693.2558,631.4092,247.3863,True


,member,weight
0,Ridge__direct,0.25
1,ElasticNet__direct,0.25
2,Ridge+LGBMResidual__direct,0.25
3,Ridge__unit,0.25


CV Champion（测试集尚未查看）: Ridge__direct


## 冻结Champion后的59炉测试审计

In [6]:
from random_cv_runtime.experiment import audit_frozen_models
random_test_audit = audit_frozen_models(random_experiment, test_df)
assert len(random_test_audit) == 19
required_test_columns = {
    'mae', 'rmse', 'wape', 'r2', 'error_gt_10pct_rate',
    'rmse_bootstrap_low95', 'rmse_bootstrap_high95',
    'interval_coverage_90', 'interval_mean_width',
}
assert required_test_columns.issubset(random_test_audit.columns)
test_rmse_winner = str(random_test_audit.sort_values('rmse').iloc[0]['candidate'])
interpretation = pd.DataFrame([
    {'role': 'CV Champion', 'candidate': selected_before_test, 'meaning': '训练集内部5折CV选出'},
    {'role': 'test RMSE winner', 'candidate': test_rmse_winner, 'meaning': '59炉测试集审计优胜者，不反向改写选模'},
])
display(random_test_audit.sort_values('rmse').round(4))
display(interpretation)
print(f'CV Champion: {selected_before_test}')
print(f'test RMSE winner: {test_rmse_winner}')
print('测试集结果不能反向改变CV Champion。')

,candidate,route,mae,rmse,wape,r2,error_gt_10pct_rate,rmse_bootstrap_low95,rmse_bootstrap_high95,interval_coverage_90,interval_mean_width,native_model_std_mean,selected_before_lock
0,Huber__direct,direct,325.7352,547.0959,0.0857,0.8931,0.2712,303.2443,803.3183,0.8814,1238.5778,NaN,False
1,Huber+LGBMResidual__direct,direct,382.9110,576.4526,0.1008,0.8813,0.3559,354.3885,820.8399,0.8814,1299.8859,NaN,False
2,ElasticNet__direct,direct,364.8377,581.6518,0.0960,0.8792,0.3220,334.8992,844.5539,0.8644,1262.6215,NaN,False
3,Ridge__direct,direct,365.8402,586.6732,0.0963,0.8771,0.3220,335.1423,854.7389,0.8644,1260.2787,NaN,True
4,OOFEnsemble,mixed_total,369.0162,595.8425,0.0971,0.8732,0.3559,336.1446,876.3378,0.8814,1270.6096,NaN,False
5,Ridge+LGBMResidual__direct,direct,388.0222,604.6021,0.1021,0.8695,0.3729,366.0028,870.9680,0.8644,1298.9450,NaN,False
6,Ridge+LGBMResidual__unit,unit,411.7097,632.2326,0.1084,0.8573,0.4068,382.6013,903.2081,0.8305,1287.7666,NaN,False
7,Huber+LGBMResidual__unit,unit,412.8225,645.1120,0.1087,0.8514,0.4407,387.8510,928.1802,0.8644,1368.3161,NaN,False
8,Ridge__unit,unit,398.5507,652.9262,0.1049,0.8478,0.3390,365.3039,959.6118,0.8644,1341.1976,NaN,False
9,Huber__unit,unit,364.7401,661.5711,0.0960,0.8437,0.3220,326.6611,1009.3939,0.8475,1268.1105,NaN,False


,role,candidate,meaning
0,CV Champion,Ridge__direct,训练集内部5折CV选出
1,test RMSE winner,Huber__direct,59炉测试集审计优胜者，不反向改写选模


CV Champion: Ridge__direct
test RMSE winner: Huber__direct
测试集结果不能反向改变CV Champion。


## 随机CV与时间滚动CV比较

In [7]:
from random_cv_runtime.validation import large_window_splits
chron_dev = df.iloc[:248].copy()
chronological_experiment = run_model_matrix(chron_dev, large_window_splits(248))
chronological_cv_summary = chronological_experiment.summary.copy()
metric_columns = ['candidate', 'rmse_mean', 'rmse_std', 'mae_mean', 'selection_score']
random_for_merge = random_cv_summary[metric_columns].rename(columns={
    column: f'random_{column}' for column in metric_columns if column != 'candidate'
})
chron_for_merge = chronological_cv_summary[metric_columns].rename(columns={
    column: f'chronological_{column}' for column in metric_columns if column != 'candidate'
})
random_vs_chronological = random_for_merge.merge(chron_for_merge, on='candidate', how='inner')
random_vs_chronological['rmse_difference_random_minus_chronological'] = (
    random_vs_chronological['random_rmse_mean']
    - random_vs_chronological['chronological_rmse_mean']
)
assert len(random_vs_chronological) == 19
display(random_vs_chronological.sort_values('random_selection_score').round(4))
print('随机划分会混合早期和晚期炉次，因此随机CV结果可能对未来炉次偏乐观。')

,candidate,random_rmse_mean,random_rmse_std,random_mae_mean,random_selection_score,chronological_rmse_mean,chronological_rmse_std,chronological_mae_mean,chronological_selection_score,rmse_difference_random_minus_chronological
0,Ridge__direct,400.3482,57.4475,302.6455,414.7101,397.5958,124.0488,304.6496,428.6080,2.7524
1,ElasticNet__direct,402.3157,55.7575,303.5668,416.2550,388.3532,119.8729,300.5711,418.3214,13.9625
2,OOFEnsemble,405.8547,55.9327,303.1860,419.8379,384.5394,118.2728,295.8505,414.1076,21.3154
3,Ridge+LGBMResidual__direct,412.4934,62.8721,302.6640,428.2114,398.0474,139.4750,300.8135,432.9161,14.4461
4,Ridge__unit,453.5601,45.2749,340.8925,464.8789,464.4812,123.3833,338.2359,495.3270,-10.9210
5,Ridge+LGBMResidual__unit,453.9626,58.2886,322.4622,468.5347,450.6692,146.2683,324.8165,487.2363,3.2933
6,GPR__direct,434.1085,138.4458,295.0731,468.7199,419.5917,150.8478,295.4180,457.3037,14.5167
7,ElasticNet__unit,458.5878,43.3705,342.7942,469.4304,452.7726,117.1311,335.6620,482.0554,5.8152
8,Huber__direct,453.1263,78.3276,306.5335,472.7082,417.8556,197.3896,301.4335,467.2030,35.2707
9,Huber+LGBMResidual__direct,460.7711,95.7990,318.6294,484.7208,414.6534,199.2812,299.9307,464.4737,46.1176


随机划分会混合早期和晚期炉次，因此随机CV结果可能对未来炉次偏乐观。


## 最终单模型重训与joblib

In [8]:
import json
import warnings
from datetime import datetime, timezone
import joblib
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
champion_result = random_experiment.results[selected_before_test]
final_champion = clone(champion_result.estimator_template)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    final_champion.fit(df[FEATURE_COLS], df[TARGET_COL])
artifact_path = ARTIFACTS_DIR / 'random_cv_champion.joblib'
bundle = {
    'artifact_version': 'random-cv-furnace-1.0',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'feature_cols': list(FEATURE_COLS),
    'target_col': TARGET_COL,
    'training_batches': int(len(df)),
    'outlier_batches_retained': outlier_count_before_split,
    'selected_model_name': selected_before_test,
    'model': final_champion,
    'cv_fold_models': champion_result.fold_models,
    'conformal_radius_90': float(champion_result.conformal_radius_90),
    'split_config': {
        'test_size': 0.20, 'random_state': 42, 'cv_folds': 5,
        'cv_shuffle': True, 'all_rows_final_refit': True,
    },
}
joblib.dump(bundle, artifact_path)
random_cv_summary.to_csv(REPORTS_DIR / 'random_cv_summary.csv', index=False)
random_cv_fold_metrics.to_csv(REPORTS_DIR / 'random_cv_fold_metrics.csv', index=False)
random_test_audit.to_csv(REPORTS_DIR / 'random_test_audit.csv', index=False)
chronological_cv_summary.to_csv(REPORTS_DIR / 'chronological_cv_summary.csv', index=False)
random_vs_chronological.to_csv(REPORTS_DIR / 'random_vs_chronological.csv', index=False)
random_experiment.tree_tuning_summary.to_csv(REPORTS_DIR / 'tree_tuning_summary.csv', index=False)
ensemble_table.to_csv(REPORTS_DIR / 'ensemble_weights.csv', index=False)
run_summary = {
    'artifact_version': bundle['artifact_version'],
    'selected_model_name': selected_before_test,
    'test_rmse_winner': test_rmse_winner,
    'training_batches': int(len(df)),
    'train_batches_for_selection': int(len(train_df)),
    'test_batches_for_audit': int(len(test_df)),
    'candidate_count': int(random_cv_summary['candidate'].nunique()),
    'artifact_path': str(artifact_path),
}
(REPORTS_DIR / 'run_summary.json').write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Champion artifact: {artifact_path}')
display(run_summary)

Champion artifact: /Users/tian/Desktop/prediction_project/advanced_furnace_ml/random_cv_output/artifacts/random_cv_champion.joblib


{'artifact_version': 'random-cv-furnace-1.0',
 'selected_model_name': 'Ridge__direct',
 'test_rmse_winner': 'Huber__direct',
 'training_batches': 292,
 'train_batches_for_selection': 233,
 'test_batches_for_audit': 59,
 'candidate_count': 19,
 'artifact_path': '/Users/tian/Desktop/prediction_project/advanced_furnace_ml/random_cv_output/artifacts/random_cv_champion.joblib'}

In [9]:
def validate_frame(input_frame, expected_columns):
    if not isinstance(input_frame, pd.DataFrame):
        raise TypeError('输入必须是pandas DataFrame。')
    expected = list(expected_columns)
    missing = [column for column in expected if column not in input_frame.columns]
    extra = [column for column in input_frame.columns if column not in expected]
    if missing:
        raise ValueError(f'缺少特征列: {missing}')
    if extra:
        raise ValueError(f'存在未声明特征列: {extra}')
    numeric = input_frame[expected].apply(pd.to_numeric, errors='coerce')
    if numeric.isna().any().any():
        bad = numeric.columns[numeric.isna().any()].tolist()
        raise ValueError(f'特征包含非数值或缺失值: {bad}')
    return numeric

def predict_bundle(loaded_bundle, input_frame):
    numeric = validate_frame(input_frame, loaded_bundle['feature_cols'])
    prediction = np.asarray(loaded_bundle['model'].predict(numeric), dtype=float)
    radius = float(loaded_bundle['conformal_radius_90'])
    fold_matrix = np.vstack([
        model.predict(numeric) for model in loaded_bundle['cv_fold_models']
    ])
    return pd.DataFrame({
        'predicted_total_gas': prediction,
        'interval_90_low': np.maximum(0.0, prediction - radius),
        'interval_90_high': prediction + radius,
        'cv_fold_prediction_std': fold_matrix.std(axis=0),
    }, index=input_frame.index)

## 最终自动测试汇总

In [10]:
loaded_bundle = joblib.load(artifact_path)
replay_input = df[FEATURE_COLS].iloc[:3].copy()
before_save = predict_bundle(bundle, replay_input)
after_load = predict_bundle(loaded_bundle, replay_input)
np.testing.assert_allclose(
    before_save.to_numpy(), after_load.to_numpy(), rtol=0.0, atol=1e-10
)
prediction_round_trip_atol_1e_10 = True

def rejects(frame):
    try:
        predict_bundle(loaded_bundle, frame)
    except (TypeError, ValueError):
        return True
    return False

missing_column_rejected = rejects(replay_input.drop(columns=[FEATURE_COLS[0]]))
extra_column_rejected = rejects(replay_input.assign(undeclared_feature=1.0))
bad_numeric = replay_input.copy()
bad_numeric.loc[bad_numeric.index[0], FEATURE_COLS[0]] = 'not-a-number'
nonnumeric_value_rejected = rejects(bad_numeric)
artifact_replay_tests = pd.DataFrame([
    {'test': 'prediction_round_trip_atol_1e_10', 'passed': prediction_round_trip_atol_1e_10},
    {'test': 'missing_column_rejected', 'passed': missing_column_rejected},
    {'test': 'extra_column_rejected', 'passed': extra_column_rejected},
    {'test': 'nonnumeric_value_rejected', 'passed': nonnumeric_value_rejected},
    {'test': 'all_292_batches_retained', 'passed': len(df) == 292},
    {'test': 'nineteen_candidates_compared', 'passed': len(random_cv_summary) == 19},
])
assert artifact_replay_tests['passed'].all(), artifact_replay_tests
display(artifact_replay_tests)
display(after_load.round(4))
print('全部自动测试通过；joblib只提供气耗预测，不包含参数推荐。')

/var/folders/7y/p8ng6rgd39g1nzlzystp076h0000gn/T/ipykernel_43579/811346724.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not-a-number' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  bad_numeric.loc[bad_numeric.index[0], FEATURE_COLS[0]] = 'not-a-number'


,test,passed
0,prediction_round_trip_atol_1e_10,True
1,missing_column_rejected,True
2,extra_column_rejected,True
3,nonnumeric_value_rejected,True
4,all_292_batches_retained,True
5,nineteen_candidates_compared,True


,predicted_total_gas,interval_90_low,interval_90_high,cv_fold_prediction_std
0,4705.3617,4075.2224,5335.5011,38.3110
1,3219.7413,2589.6019,3849.8806,50.1456
2,4323.6988,3693.5595,4953.8381,37.9359


全部自动测试通过；joblib只提供气耗预测，不包含参数推荐。
